# Impulse-Response Experiments

Dedicated paired baseline-versus-shock macro workflow. The implementation lives in `src/experiments/irf.py`; the historical command remains available through `run_irf_experiment.py`. GFCF records gross purchases of capital goods, including replacement and additions above replacement; GFCF intensity is aggregate GFCF divided by nominal GDP.

USES BENCHMARK CONFIG

In [13]:
# Setup
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd
from src.experiments.irf import run_irf_experiment
from src.visual_helpers import plot_irf_panels

from macromodel.utils.prehooks.irf_shocks import ShockSpec

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Inputs

In [14]:
RUN_IRF = True
COUNTRY = "FRA"
SEEDS = list(range(1, 50))
HORIZON_PERIODS = 25
SHOCK_PERIOD = 50
T_MAX = 75
N_JOBS = 4
# Monetary-policy regime for this experiment only (YAML unchanged).
# "elb": estimated rule with the zero floor (benchmark; the rate stays at the floor at baseline).
# "no_elb": same estimated coefficients, floor removed: the rule responds symmetrically.
POLICY_REGIME = "no_elb"
POLICY_RULE_OVERRIDES = {"elb": None, "no_elb": {"lower_bound": None}}[POLICY_REGIME]
PREVENT_REHIRING = True
REHIRING_SCOPE = "all_unemployed"  # Pause all hiring from unemployment; "displaced" restricts only shock cohorts.
OUTPUT_DIR = Path("data/output_data/irf-experiment" + ("" if POLICY_REGIME == "elb" else f"-{POLICY_REGIME}"))
OUTPUT_DIR = OUTPUT_DIR.with_name(OUTPUT_DIR.name + "-job-loss-designs")
if PREVENT_REHIRING:
    OUTPUT_DIR = OUTPUT_DIR.with_name(OUTPUT_DIR.name + "-no-rehiring")
    if REHIRING_SCOPE == "all_unemployed":
        OUTPUT_DIR = OUTPUT_DIR.with_name(OUTPUT_DIR.name + "-all-unemployed")

shock_specs = (
    ShockSpec(
        name="gov_cons_5pct",
        kind="government_consumption",
        period=SHOCK_PERIOD,
        magnitude=0.05,
        duration=4,
        mode="multiplicative",
    ),
    # Monetary tightening, headline: one +1pp (annual) innovation to the Taylor rule. The rate jumps 1pp
    # on impact and decays through smoothing (rho = 0.904, half-life about 7 quarters), as in DSGE/HANK.
    ShockSpec(
        name="policy_rate_1pp_once", kind="policy_rate", period=SHOCK_PERIOD, magnitude=0.01, duration=1, mode="additive"
    ),
    # Robustness, gradual hiking cycle: four +0.29pp innovations. Smoothing accumulates them, so the rate
    # path is about 0.29, 0.55, 0.79, 1.00pp at h0..h3 (peak 0.29 * (1 + rho + rho^2 + rho^3) = 1pp).
    ShockSpec(
        name="policy_rate_gradual_1pp",
        kind="policy_rate",
        period=SHOCK_PERIOD,
        magnitude=0.0029,
        duration=4,
        mode="additive",
    ),
    ShockSpec(
        name="income_tax_5pp", kind="income_tax", period=SHOCK_PERIOD, magnitude=0.05, duration=4, mode="additive"
    ),
    ShockSpec(
        # One-off 2% separations; the selected hiring restriction lasts four quarters.
        name="unemp_2pp_once",
        kind="unemployment_rate",
        period=SHOCK_PERIOD,
        magnitude=0.02,
        duration=1,
        mode="additive",
        prevent_rehiring=PREVENT_REHIRING,
        rehiring_scope=REHIRING_SCOPE if PREVENT_REHIRING else "displaced",
        rehiring_lockout_duration=4 if PREVENT_REHIRING else None,
    ),
    ShockSpec(
        # Milder variant: one-off 1% separations, same four-quarter hiring restriction.
        name="unemp_1pp_once",
        kind="unemployment_rate",
        period=SHOCK_PERIOD,
        magnitude=0.01,
        duration=1,
        mode="additive",
        prevent_rehiring=PREVENT_REHIRING,
        rehiring_scope=REHIRING_SCOPE if PREVENT_REHIRING else "displaced",
        rehiring_lockout_duration=4 if PREVENT_REHIRING else None,
    ),
    ShockSpec(
        # Alternative: four 0.5% separation flows, with the same four-quarter hiring restriction.
        name="unemp_05pp_quarterly",
        kind="unemployment_rate",
        period=SHOCK_PERIOD,
        magnitude=0.005,
        duration=4,
        mode="additive",
        prevent_rehiring=PREVENT_REHIRING,
        rehiring_scope=REHIRING_SCOPE if PREVENT_REHIRING else "displaced",
        rehiring_lockout_duration=4 if PREVENT_REHIRING else None,
    ),
    # Common and individual uncertainty shocks dropped: common is inert (tiny variance), individual is
    # a subset of the total shock below.
    # One-off government transfer to every household, same size and channel as the MPC experiment
    # (10% of median positive household income, exempt from the support test, booked in the deficit).
    ShockSpec(
        name="household_transfer_10pct",
        kind="household_transfer",
        period=SHOCK_PERIOD,
        magnitude=0.10,
        mode="multiplicative",
    ),
    # Raises the total income-uncertainty index theta (individual + common) by 1% for 4 quarters.
    # Size: d(theta)/du = 0.0070 per unit of the unemployment rate (vault row 22) and theta ~ 0.008,
    # so +1% of theta ~ the uncertainty change of a +1pp unemployment rise (Aron et al. scale).
    # The rule responds to the change in theta: expect a cut at onset and a rebound when it ends (h4).
    ShockSpec(
        name="total_income_uncertainty_1pct",
        kind="total_income_uncertainty",
        period=SHOCK_PERIOD,
        magnitude=0.01,
        duration=4,
        mode="multiplicative",
    ),
)

# Explicit current outputs; remove names here to save fewer series.
OUTPUT_VARIABLES = [
    "household_consumption",
    "target_household_consumption",
    "household_income",
    "government_consumption",
    "desired_government_consumption",
    "policy_rate",
    "cpi",
    "gdp_output",
    "gdp_expenditure",
    "gdp_income",
    "unemployment_rate",
    "gross_fixed_capital_formation",
    "firm_production",
    "firm_employees",
    "firm_wage_per_employee",
    "household_real_employee_income",
    "income_uncertainty_delta",
    "consumption_uncertainty_contribution",
    "firm_loan_interest_paid",
    "firm_received_credit",
    "firm_debt",
    "firm_deposits",
    "household_deposits",
    "household_employee_income",
    "household_social_transfers",
    "household_unemployment_benefits",
    "household_consumption_loan_interest",
    "household_deposit_interest",
    "household_employment",
    "household_consumption_loan_rate",
    "household_mortgage_rate",
    "household_deposit_rate",
    "firm_deposit_rate",
    "tfp_investment_intensity",
    "gfcf_intensity",
    "firm_loan_rate",
    "firm_long_term_loan_rate",
]

## Run

In [15]:
outputs = None
if RUN_IRF:
    outputs = run_irf_experiment(
        output_variables=OUTPUT_VARIABLES,
        seeds=SEEDS,
        t_max=T_MAX,
        shock_specs=shock_specs,
        horizon_periods=HORIZON_PERIODS,
        output_dir=OUTPUT_DIR,
        country_iso3=COUNTRY,
        n_jobs=N_JOBS,
        force_rebuild_data=True,
        policy_rule_overrides=POLICY_RULE_OVERRIDES,
    )
else:
    print("Set RUN_IRF = True to run the experiment.")

{'seed': 1, 'country': 'FRA', 't_max': 75, 'raw_data_path': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/raw_data', 'output_dir': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data/irf-experiment-no_elb-job-loss-designs-no-rehiring-all-unemployed', 'data_cache': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data/irf-experiment-no_elb-job-loss-designs-no-rehiring-all-unemployed/data.pkl'}
Configuration summary
{'productivity_growth': 'SimpleTFPGrowth',
 'productivity_investment_planner': 'TargetIntensityTFPInvestmentPlanner',
 'labour_market': {'name': 'ReservationWageBindingDefaultLabourMarketClearer',
                   'parameters': {'allow_switching_industries': True,
                                  'compare_with_normalised_inputs': True,
                                  'consider_reservation_wages': True,
                                  'firing_cost_fraction': 0.0,
                 

/Users/andone/Documents/python_projects/INET-consumption/macromodel/agents/firms/func/prices.py:734: RuntimeWarning: Markup residual calibration: 2 sector(s) have an AC floor above their initial-price anchor and cannot reach it at any residual factor: industry 1 (+3.36%), industry 11 (+4.65%). These sectors keep their unadjusted Orbis markup and price at average cost until unit depreciation falls relative to marginal cost. The percentage is the AC-floor overshoot of the anchor, also recorded as pricing_markup_residual_unreachable_gap.
  self._warn_on_unreachable_markup_anchors()
/Users/andone/Documents/python_projects/INET-consumption/macromodel/agents/firms/func/prices.py:734: RuntimeWarning: Markup residual calibration: 2 sector(s) have an AC floor above their initial-price anchor and cannot reach it at any residual factor: industry 1 (+3.36%), industry 11 (+4.65%). These sectors keep their unadjusted Orbis markup and price at average cost until unit depreciation falls relative to ma

RuntimeError: Consumer-credit household liabilities and bank assets were not booked exactly once.

## Plots

In [ ]:
if outputs is not None:
    panel = pd.read_csv(outputs["analysis_dir"] / "irf_panel.csv")
    plot_irf_panels(panel, [s.name for s in shock_specs], country=COUNTRY, color="blue", periods_per_year=4, confidence=0.10, inner_confidence=0.25)

In [ ]:
from macro_data.configuration.countries import Country
from macro_data.processing.country_data.tax_data import TaxData
from macro_data.readers.default_readers import DataPaths
from macro_data.readers.economic_data.eurostat_reader import EuroStatReader
from macro_data.readers.economic_data.oecd_economic_data import OECDEconData
from macro_data.readers.economic_data.world_bank_reader import WorldBankReader

# Select any country and year available in the raw reader data.
TAX_COUNTRY = COUNTRY
TAX_YEAR = 2010

country = Country(TAX_COUNTRY)
raw_data_path = Path("data/raw_data")
paths = DataPaths.default_paths(raw_data_path, [TAX_YEAR])
world_bank = WorldBankReader(paths.world_bank_path)
oecd_econ = OECDEconData(paths.oecd_econ_path, scale_dict={country: 1})
eurostat = EuroStatReader(paths.eurostat_path, paths.country_codes_path)


class TaxReaders:
    world_bank = world_bank
    oecd_econ = oecd_econ
    eurostat = eurostat

    @staticmethod
    def get_export_taxes(country, year):
        return world_bank.get_tau_exp(country, year)


readers = TaxReaders()


def tax_data_from_readers(readers, country, year):
    return TaxData(
        value_added_tax=readers.world_bank.get_tau_vat(country, year),
        export_tax=readers.get_export_taxes(country, year),
        employer_social_insurance_tax=readers.oecd_econ.read_tau_sif(country, year),
        employee_social_insurance_tax=readers.oecd_econ.read_tau_siw(country, year),
        profit_tax=readers.oecd_econ.read_tau_firm(country, year),
        income_tax=readers.oecd_econ.read_tau_income(country, year),
        risk_premium=readers.eurostat.firm_risk_premium(country, year),
        capital_formation_tax=readers.eurostat.taxrate_on_capital_formation(country, year),
    )


tax_data = tax_data_from_readers(readers, country, TAX_YEAR)

tax_rates = {
    "VAT": tax_data.value_added_tax,
    "Export tax": tax_data.export_tax,
    "Employer social insurance": tax_data.employer_social_insurance_tax,
    "Employee social insurance": tax_data.employee_social_insurance_tax,
    "Profit tax": tax_data.profit_tax,
    "Income tax": tax_data.income_tax,
    "Capital formation tax": tax_data.capital_formation_tax,
}

for name, rate in tax_rates.items():
    print(f"{name}: {rate:.6%}")